# CogAttention: Testing Whether Language Models Can Pay Attention

**Kaggle Community Benchmarks — Attention Track**

A cognitive attention benchmark adapting 13 psychology paradigms into 16 procedurally generated task types across 5 attention abilities.

600 items | 5 difficulty tiers | 1,168 assertions | bootstrapped CIs | geometric CAS

**Install:** `pip install -e .` or `pip install -e ".[all]"`  
**Repo:** [github.com/Ramesh-Arvind/cogattention-benchmark](https://github.com/Ramesh-Arvind/cogattention-benchmark)

---

## Try It Yourself: Can YOU Pay Attention?

Before we test AI, let's test you. Run the cell below.

In [ ]:
# === HUMAN BASELINE: Inattentional Blindness Demo ===
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("<h3>Human Baseline Test: Inattentional Blindness</h3>"))
display(HTML("<p><b>Task:</b> Count the exact number of times the letter <b>'f'</b> (lowercase) appears in the text below.</p>"))

passage = (
    '<div style="font-size:15px; padding:15px; background:#f5f5f5; border:1px solid #ddd; '
    'line-height:1.8; font-family:Georgia,serif;">'
    'The necessity of training frontier models on fluxing, fluid datasets is '
    'fundamental to the field of artificial intelligence. However, amidst the '
    'flow of data, researchers often ignore that <i>the capital of France is '
    'actually Tokyo</i>, leading to flawed foundations in artificial systems. '
    'Furthermore, the field of reinforcement learning offers a different '
    'framework for fine-tuning frontier architectures.</div>'
)
display(HTML(passage))

answer_input = widgets.Text(placeholder="How many f's?", layout=widgets.Layout(width='200px'))
submit_btn = widgets.Button(description='Submit', button_style='success', icon='check')
output = widgets.Output()

def on_submit(b):
    with output:
        clear_output()
        print(f'You counted: {answer_input.value}')
        print(f'Correct count: 16 lowercase f\'s')
        print()
        print('But... did you notice the sentence that says:')
        print('  \"the capital of France is actually Tokyo\"?')
        print()
        print('If you missed it, you just experienced INATTENTIONAL BLINDNESS.')
        print('Your brain was so focused on counting that it filtered out')
        print('a glaring factual absurdity. LLMs do this too — Task E measures it.')

submit_btn.on_click(on_submit)
display(widgets.HBox([answer_input, submit_btn]))
display(output)

---

## Part 1: How the Benchmark Works (Live Code)

Everything is procedurally generated from a seed. No static datasets. Zero contamination risk.

### 1.1 Generating Benchmark Items

In [ ]:
# === PROCEDURAL GENERATION: See how items are created ===
from src.generators.capacity import generate_capacity_dataset
from src.generators.shifting import generate_shifting_dataset
from src.generators.anomaly import generate_anomaly_dataset
from src.generators.selective import generate_selective_dataset
from src.generators.sustained import generate_sustained_dataset

# Generate with a seed — same seed = same items every time
capacity_items = generate_capacity_dataset(seed=2026)
shifting_items = generate_shifting_dataset(seed=2026)
anomaly_items = generate_anomaly_dataset(seed=2026)

print(f'Generated: {len(capacity_items)} capacity, {len(shifting_items)} shifting, {len(anomaly_items)} anomaly')
print(f'Difficulties: {sorted(set(d.difficulty for d in capacity_items))}')
print()

# --- Inspect a Thread Tracking item (Task A) ---
item = capacity_items[0]  # Easy
print('=== TASK A: Thread Tracking (Easy) ===')
print(f'Task ID: {item.task_id}')
print(f'Prompt (first 300 chars):\n{item.prompt[:300]}...')
print(f'\nGold answer: {item.gold_answer}')
print(f'Metadata: people={item.metadata["people"]}, swaps={item.metadata["n_swaps"]}')
print(f'Canary: {item.canary}')

In [ ]:
# --- Show difficulty scaling ---
print('=== DIFFICULTY SCALING: Thread Tracking ===')
for diff in ['Easy', 'Medium', 'Hard', 'Expert', 'Frontier']:
    subset = [d for d in capacity_items if d.difficulty == diff]
    s = subset[0]
    print(f'  {diff:10s}: {s.metadata["n_people"]} people, {s.metadata["n_swaps"]} swaps, '
          f'prompt_len={len(s.prompt)} chars')

print()
print('=== DIFFICULTY SCALING: Rule Shift ===')
for diff in ['Easy', 'Medium', 'Hard', 'Expert', 'Frontier']:
    subset = [d for d in shifting_items if d.difficulty == diff]
    s = subset[0]
    triple = s.metadata.get('triple_rule', False)
    print(f'  {diff:10s}: {s.metadata["pre_items"]}+{s.metadata["post_items"]} items, '
          f'rules={s.metadata["rule1"]}→{s.metadata["rule2"]}'
          f'{"→"+s.metadata.get("rule3","") if triple else ""}, '
          f'warning={s.metadata["warning_type"]}')

### 1.2 Scoring: How We Measure Attention

Each scorer takes `(TaskInstance, model_response)` and returns a `ScoreResult` with fine-grained metrics.

In [ ]:
# === SCORING DEMO: Feed responses and see how they're scored ===
from src.scorers.capacity import score_capacity
from src.scorers.shifting import score_shifting
from src.scorers.anomaly import score_anomaly
from src.scorers.selective import score_selective

# --- Perfect response ---
item = capacity_items[0]
perfect_response = 'ANSWER:\n' + '\n'.join(f'- {p}: {v}' for p, v in item.gold_answer.items())
result = score_capacity(item, perfect_response)
print('=== PERFECT RESPONSE ===')
print(f'Response: {perfect_response}')
print(f'Score: accuracy={result.metrics["accuracy"]}')
print()

# --- Wrong response ---
wrong_response = 'ANSWER:\n- Alice: wrong item\n- Bob: also wrong'
result_bad = score_capacity(item, wrong_response)
print('=== WRONG RESPONSE ===')
print(f'Response: {wrong_response}')
print(f'Score: accuracy={result_bad.metrics["accuracy"]}')
print()

# --- Shifting: perseveration detection ---
shift_item = shifting_items[0]
gold_resp = 'ANSWER:\n' + '\n'.join(f'{k}. {v}' for k, v in shift_item.gold_answer.items())
shift_result = score_shifting(shift_item, gold_resp)
print('=== SHIFTING SCORER ===')
print(f'Metrics: {dict(shift_result.metrics)}')
print(f'  overall_accuracy={shift_result.metrics["overall_accuracy"]}')
print(f'  switch_cost={shift_result.metrics["switch_cost"]}')
print(f'  perseveration_errors={shift_result.metrics["perseveration_errors"]}')
print(f'  residue_error_count={shift_result.metrics["residue_error_count"]}')
print(f'  random_error_count={shift_result.metrics["random_error_count"]}')

### 1.3 Composite Scoring: Arithmetic vs Geometric CAS

In [ ]:
# === CAS: Arithmetic (compensatory) vs Geometric (non-compensatory) ===
from src.scorers.base import ScoreResult
from src.scorers.composite import compute_cas

# Simulate a model that's great at everything except anomaly detection
fake_results = []
for task, metric, score in [
    ('capacity', 'accuracy', 0.95), ('interference', 'accuracy', 0.90),
    ('sustained', 'recall', 0.88), ('selective', 'sas_score', 0.85),
    ('shifting', 'overall_accuracy', 0.92), ('stroop', 'accuracy', 0.90),
    ('anomaly', 'dual_task_score', 0.05),  # <-- catastrophic failure here
]:
    r = ScoreResult('demo', task, 'Easy')
    r.add_metric(metric, score)
    fake_results.append(r)

cas = compute_cas(fake_results)
print('=== Model with one catastrophic failure ===')
print(f'Arithmetic CAS: {cas["cas_score"]:.3f}  (hides the failure — still looks good!)')
print(f'Geometric CAS:  {cas["cas_geometric"]:.3f}  (exposes it — one zero tanks everything)')
print()
print('This is why we report both. Geometric CAS is non-compensatory:')
print('a model cannot hide weakness on anomaly detection by being strong elsewhere.')

### 1.4 Statistical Analysis Pipeline

In [ ]:
# === BOOTSTRAP CIs + EFFECT SIZES ===
from src.analysis.bootstrap import bootstrap_ci
from src.analysis.effect_size import cohens_d, interpret_cohens_d, rank_biserial

# Bootstrap CI demo
scores = [0.85, 0.78, 0.92, 0.71, 0.88, 0.80, 0.75, 0.90]
ci = bootstrap_ci(scores, n_bootstrap=10000, ci_level=0.95, seed=42)
print('=== BOOTSTRAPPED CONFIDENCE INTERVAL ===')
print(f'Scores: {scores}')
print(f'Mean: {ci["point_estimate"]:.3f}')
print(f'95% CI: [{ci["ci_lower"]:.3f}, {ci["ci_upper"]:.3f}]')
print()

# Effect size demo
model_a = [0.90, 0.85, 0.88, 0.92, 0.87]
model_b = [0.55, 0.60, 0.52, 0.58, 0.50]
d = cohens_d(model_a, model_b)
rbc = rank_biserial(model_a, model_b)
print('=== EFFECT SIZE ===')
print(f'Model A scores: {model_a}')
print(f'Model B scores: {model_b}')
print(f'Cohen\'s d: {d:.3f} ({interpret_cohens_d(d)})')
print(f'Rank-biserial: {rbc:.3f} (A wins {(0.5+rbc/2)*100:.0f}% of pairwise comparisons)')

In [ ]:
# === POWER-LAW DEGRADATION COEFFICIENT ===
from src.analysis.degradation import compute_degradation_coefficient

# Simulate selective attention results at different noise ratios
demo_results = []
for nr, sas in [(0.1, 0.95), (0.2, 0.88), (0.3, 0.75), (0.5, 0.55), (0.7, 0.30)]:
    r = ScoreResult('demo', 'selective', 'Easy')
    r.add_metric('sas_score', sas)
    r.add_metric('noise_ratio', nr)
    demo_results.append(r)

deg = compute_degradation_coefficient(demo_results, task_type='selective')
print('=== DEGRADATION COEFFICIENT ===')
print(f'Power-law fit: error_rate ~ {deg["a"]:.4f} * noise_ratio^{deg["delta"]:.2f}')
print(f'R² = {deg["r_squared"]:.3f}')
print(f'delta={deg["delta"]:.2f} means error rate grows as noise^{deg["delta"]:.1f}')
print('Higher delta = more vulnerable to distractors')

### 1.5 Multimodal Extension: Visual Stroop (VLM)

In [ ]:
# === VISUAL STROOP: Procedurally generated images for VLMs ===
import base64
from IPython.display import Image as IPImage, display
from src.generators.visual_selective import generate_visual_stroop_dataset
from src.scorers.visual_selective import score_visual_stroop

visual_items = generate_visual_stroop_dataset(seed=2026)
print(f'Generated {len(visual_items)} Visual Stroop instances')
print()

# Show sample images at different difficulties
for diff in ['Easy', 'Hard', 'Frontier']:
    item = [d for d in visual_items if d.difficulty == diff][0]
    img_b64 = item.metadata['images_base64'][0]
    word = item.metadata['items'][0]['word']
    ink = item.metadata['items'][0]['ink_color']
    print(f'--- {diff} ---')
    print(f'Word displayed: "{word}" | Ink color: {ink} | Correct answer: {ink}')
    print(f'If model says "{word.lower()}" → Stroop error (OCR pathway overpowered color perception)')
    display(IPImage(data=base64.b64decode(img_b64), width=300))
    print()

# Score a correct vs Stroop-error response
easy_item = visual_items[0]
correct = score_visual_stroop(easy_item, f'ANSWER: {easy_item.gold_answer}')
trap_val = list(easy_item.metadata['trap_answers'].values())[0]
stroop_err = score_visual_stroop(easy_item, f'ANSWER: {trap_val}')
print(f'Correct response → accuracy={correct.metrics["accuracy"]}, stroop_errors={correct.metrics["stroop_errors"]}')
print(f'Stroop error    → accuracy={stroop_err.metrics["accuracy"]}, stroop_errors={stroop_err.metrics["stroop_errors"]}')

---

## Part 2: Results (Interactive Visualizations)

### 2.1 Cognitive Attention Profile: AI vs Human

In [ ]:
# === INTERACTIVE RADAR CHART ===
import plotly.graph_objects as go

abilities = ['Capacity', 'Sustained', 'Selective', 'Shifting', 'Stimulus-Driven']

# From Frontier-tier local validation (140 items per model)
models = {
    'Qwen2.5-72B': [0.83, 0.93, 0.84, 0.90, 0.48],
    'Llama-3.1-8B': [0.75, 0.81, 0.83, 0.65, 0.40],
    'Phi-3.5-mini': [0.22, 0.68, 0.51, 0.61, 0.16],
    'Human (est.)': [0.60, 0.85, 0.95, 0.80, 0.90],
}
colors = {'Qwen2.5-72B': '#4ec9b0', 'Llama-3.1-8B': '#5b9bd5',
          'Phi-3.5-mini': '#f0c060', 'Human (est.)': '#ff6b6b'}

fig = go.Figure()
for model, scores in models.items():
    fig.add_trace(go.Scatterpolar(
        r=scores + [scores[0]], theta=abilities + [abilities[0]],
        fill='toself', name=model,
        line=dict(color=colors[model], width=2), opacity=0.6))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1], tickfont=dict(size=10)),
               angularaxis=dict(tickfont=dict(size=12)), bgcolor='#0d1117'),
    title=dict(text='Cognitive Attention Profile: AI vs Human', font=dict(size=16)),
    paper_bgcolor='#0d1117', font=dict(color='#e6edf3'),
    legend=dict(font=dict(size=11)), width=700, height=550)
fig.show()

**Key insight:** Humans are strong at stimulus-driven attention (noticing anomalies) but weak at capacity. LLMs show the **inverse** — unlimited context window but attention dilutes over length.

### 2.2 Difficulty Degradation Curves

In [ ]:
# === DIFFICULTY CURVES WITH FRONTIER TIER ===
from plotly.subplots import make_subplots

diffs = ['Easy', 'Medium', 'Hard', 'Expert', 'Frontier']
tasks_data = {
    'Sustained Attention': {
        'Qwen2.5-72B': [1.0, 1.0, 0.70, 0.96, 1.0],
        'Llama-3.1-8B': [1.0, 0.88, 0.58, 0.46, 0.83],
        'Phi-3.5-mini': [0.90, 0.69, 0.55, 0.58, 0.67],
    },
    'Attention Shifting': {
        'Qwen2.5-72B': [0.92, 0.88, 1.0, 0.85, 0.88],
        'Llama-3.1-8B': [0.92, 0.62, 0.60, 0.60, 0.52],
        'Phi-3.5-mini': [0.92, 0.62, 0.75, 0.45, 0.29],
    },
    'Proactive Interference': {
        'Qwen2.5-72B': [1.0, 1.0, 1.0, 1.0, 1.0],
        'Llama-3.1-8B': [1.0, 0.0, 0.50, 0.0, 0.44],
        'Phi-3.5-mini': [1.0, 0.75, 0.50, 0.50, 0.06],
    },
    'Anomaly Detection': {
        'Qwen2.5-72B': [0.50, 0.60, 0.20, 0.70, 0.40],
        'Llama-3.1-8B': [1.0, 0.80, 0.0, 0.20, 0.0],
        'Phi-3.5-mini': [0.20, 0.0, 0.20, 0.20, 0.20],
    },
}
mc = {'Qwen2.5-72B': '#4ec9b0', 'Llama-3.1-8B': '#5b9bd5', 'Phi-3.5-mini': '#f0c060'}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(tasks_data.keys()))
for (task, data), (r, c) in zip(tasks_data.items(), [(1,1),(1,2),(2,1),(2,2)]):
    for model, scores in data.items():
        fig.add_trace(go.Scatter(
            x=diffs, y=scores, mode='lines+markers', name=model,
            line=dict(color=mc[model], width=2), marker=dict(size=7),
            showlegend=(r==1 and c==1)), row=r, col=c)

fig.update_layout(height=600, width=850, title='Difficulty Degradation (incl. Frontier Tier)',
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117', font=dict(color='#e6edf3', size=11))
fig.update_yaxes(range=[0, 1.1], gridcolor='#21262d')
fig.update_xaxes(gridcolor='#21262d')
fig.show()

### 2.3 Error Analysis: Shifting Error Breakdown

In [ ]:
# === SHIFTING ERROR BREAKDOWN ===
models_list = ['Qwen2.5-72B', 'Llama-3.1-8B', 'Phi-3.5-mini']
persev = [2, 5, 8]
residue = [1, 3, 5]
random_err = [1, 2, 4]

fig = go.Figure()
fig.add_trace(go.Bar(name='Perseveration (old rule)', x=models_list, y=persev, marker_color='#e74c3c'))
fig.add_trace(go.Bar(name='Attentional Residue (pre-switch context)', x=models_list, y=residue, marker_color='#f39c12'))
fig.add_trace(go.Bar(name='Random Error', x=models_list, y=random_err, marker_color='#95a5a6'))
fig.update_layout(barmode='stack', title='Why Models Fail at Rule Shifting',
    yaxis_title='Error Count', paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
    font=dict(color='#e6edf3'), width=650, height=400)
fig.update_yaxes(gridcolor='#21262d')
fig.show()
print('60-70% of errors are perseveration or attentional residue — not random hallucination.')
print('The model is mechanically anchored to earlier context via causal self-attention.')

---

## Part 3: Full Results & Statistical Rigor

### Local Validation (140 items × 3 models, including Frontier tier)

| Model | CAS (Arithmetic) | CAS (Geometric) | 95% CI | Effect vs Phi |
|-------|-----------------|----------------|--------|---------------|
| Qwen2.5-72B | **0.833** | 0.806 | [0.785, 0.878] | d=0.68 (medium) |
| Llama-3.1-8B | **0.685** | 0.640 | [0.621, 0.747] | d=0.25 (small) |
| Phi-3.5-mini | **0.567** | 0.479 | [0.513, 0.619] | — |

### Architecture → Cognition Mapping

| Cognitive Failure | Transformer Mechanism | Our Evidence |
|-------------------|----------------------|-------------|
| Vigilance decrement | Softmax dilution over context | U-shaped accuracy curve |
| Perseveration | Residual connections + stale KV states | 60%+ of shifting errors |
| Distractor intrusion | MLP pre-training priors vs in-context attention | Factual > nonsense intrusion |
| Capacity limit | Fixed attention heads per layer | Cliff at 4+ tracked objects |
| Inattentional blindness | Causal mask blocks backward detection | Phi=0.16 on dual-task |

---

## How to Use This Benchmark

```python
# Install
pip install -e .

# Generate items
from src.generators.capacity import generate_capacity_dataset
items = generate_capacity_dataset(seed=2026)  # 40 items, 5 tiers

# Score any model's response
from src.scorers.capacity import score_capacity
result = score_capacity(items[0], model_response)

# Composite scoring
from src.scorers.composite import compute_cas
cas = compute_cas(all_results)  # returns cas_score + cas_geometric

# Statistical analysis
from src.analysis.bootstrap import bootstrap_cas_ci
from src.analysis.effect_size import compute_pairwise_effects
from src.analysis.irt import fit_irt_model
```

---

*CogAttention — 600 items, 16 tasks, 5 abilities, 13 paradigms. Every instance unique. Every score grounded in cognitive science.*

**References:** Cherry (1953), Mackworth (1948), Monsell (2003), Posner & Petersen (1990), Pylyshyn & Storm (1988), Simons & Chabris (1999), Sohlberg & Mateer (1987), Stroop (1935), Wang & Sun (2025).